# Lab | Hypothesis Testing

**Objective**

Welcome to the Hypothesis Testing Lab, where we embark on an enlightening journey through the realm of statistical decision-making! In this laboratory, we delve into various scenarios, applying the powerful tools of hypothesis testing to scrutinize and interpret data.

From testing the mean of a single sample (One Sample T-Test), to investigating differences between independent groups (Two Sample T-Test), and exploring relationships within dependent samples (Paired Sample T-Test), our exploration knows no bounds. Furthermore, we'll venture into the realm of Analysis of Variance (ANOVA), unraveling the complexities of comparing means across multiple groups.

So, grab your statistical tools, prepare your hypotheses, and let's embark on this fascinating journey of exploration and discovery in the world of hypothesis testing!

**Challenge 1**

In this challenge, we will be working with pokemon data. The data can be found here:

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv

In [1]:
#libraries
import pandas as pd
import scipy.stats as st
import numpy as np



In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")
df.head()

,Name,Type 1,Type 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,Bulbasaur,Grass,Poison,45,49,49,65,65,45,1,False
1,Ivysaur,Grass,Poison,60,62,63,80,80,60,1,False
2,Venusaur,Grass,Poison,80,82,83,100,100,80,1,False
3,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,1,False
4,Charmander,Fire,NaN,39,52,43,60,50,65,1,False


- We posit that Pokemons of type Dragon have, on average, more HP stats than Grass. Choose the propper test and, with 5% significance, comment your findings.

In [4]:
dragon_hp = df.loc[df['Type 1'] == 'Dragon', 'HP']
grass_hp = df.loc[df['Type 1'] == 'Grass', 'HP']

print("Media de HP de Dragon:", dragon_hp.mean())
print("Media de HP de Grass:", grass_hp.mean())

Media de HP de Dragon: 83.3125
Media de HP de Grass: 67.27142857142857


In [5]:
statistic, p_value = st.ttest_ind(
    dragon_hp,
    grass_hp,
    equal_var=False,
    alternative='greater'
)

print("Estadístico:", statistic)
print("P-valor:", p_value)

Estadístico: 3.3349632905124063
P-valor: 0.0007993609745420602


Con un nivel de significación del 5 %, el t-test de Welch unilateral proporciona evidencia de que los Pokémon cuyo tipo principal es Dragon tienen una media de HP superior a los de tipo Grass. En los datos analizados, las medias son 83,31 y 67,27 HP, respectivamente, una diferencia de 16,04 HP.

- We posit that Legendary Pokemons have different stats (HP, Attack, Defense, Sp.Atk, Sp.Def, Speed) when comparing with Non-Legendary. Choose the propper test and, with 5% significance, comment your findings.


In [6]:
estadisticas = ['HP', 'Attack', 'Defense', 'Sp. Atk', 'Sp. Def', 'Speed']

for columna in estadisticas:
    legendarios = df.loc[df['Legendary'] == True, columna]
    no_legendarios = df.loc[df['Legendary'] == False, columna]

    statistic, p_value = st.ttest_ind(
        legendarios,
        no_legendarios,
        equal_var=False,
        alternative='two-sided'
    )

    print("Estadística:", columna)
    print("Media legendarios:", legendarios.mean())
    print("Media no legendarios:", no_legendarios.mean())
    print("P-valor:", p_value)

Estadística: HP
Media legendarios: 92.73846153846154
Media no legendarios: 67.18231292517007
P-valor: 1.0026911708035289e-13
Estadística: Attack
Media legendarios: 116.67692307692307
Media no legendarios: 75.66938775510204
P-valor: 2.520372449236657e-16
Estadística: Defense
Media legendarios: 99.66153846153846
Media no legendarios: 71.55918367346939
P-valor: 4.8269984949193406e-11
Estadística: Sp. Atk
Media legendarios: 122.18461538461538
Media no legendarios: 68.45442176870748
P-valor: 1.5514614112239793e-21
Estadística: Sp. Def
Media legendarios: 105.93846153846154
Media no legendarios: 68.89251700680272
P-valor: 2.2949327864052933e-15
Estadística: Speed
Media legendarios: 100.18461538461538
Media no legendarios: 65.45578231292517
P-valor: 1.049016311882455e-18


Los t-tests de Welch bilaterales muestran diferencias estadísticamente significativas entre Pokémon legendarios y no legendarios en las seis estadísticas analizadas. Los legendarios presentan medias superiores en todas ellas. La mayor diferencia en puntos se encuentra en el ataque especial: 122,18 frente a 68,45.
Aunque se realizan seis pruebas, los p-valores son tan pequeños que las conclusiones se mantienen incluso aplicando una corrección de Bonferroni, cuyo umbral sería 0,05 / 6 ≈ 0,0083.

**Challenge 2**

In this challenge, we will be working with california-housing data. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv

In [7]:
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0


**We posit that houses close to either a school or a hospital are more expensive.**

- School coordinates (-118, 34)
- Hospital coordinates (-122, 37)

We consider a house (neighborhood) to be close to a school or hospital if the distance is lower than 0.50.

Hint:
- Write a function to calculate euclidean distance from each house (neighborhood) to the school and to the hospital.
- Divide your dataset into houses close and far from either a hospital or school.
- Choose the propper test and, with 5% significance, comment your findings.
 

In [8]:
def distancia_euclidea(longitud, latitud, punto_longitud, punto_latitud):
    return np.sqrt(
        (longitud - punto_longitud) ** 2 +
        (latitud - punto_latitud) ** 2
    )

df['distancia_escuela'] = distancia_euclidea(
    df['longitude'], df['latitude'], -118, 34
)

df['distancia_hospital'] = distancia_euclidea(
    df['longitude'], df['latitude'], -122, 37
)

In [9]:
df['cerca'] = (
    (df['distancia_escuela'] < 0.50) |
    (df['distancia_hospital'] < 0.50)
)

precios_cerca = df.loc[df['cerca'], 'median_house_value'].dropna()
precios_lejos = df.loc[~df['cerca'], 'median_house_value'].dropna()

print("Zonas cercanas:", len(precios_cerca))
print("Zonas alejadas:", len(precios_lejos))
print("Precio medio en zonas cercanas:", precios_cerca.mean())
print("Precio medio en zonas alejadas:", precios_lejos.mean())

Zonas cercanas: 6829
Zonas alejadas: 10171
Precio medio en zonas cercanas: 246951.98213501245
Precio medio en zonas alejadas: 180678.44105790975


In [10]:
statistic, p_value = st.ttest_ind(
    precios_cerca,
    precios_lejos,
    equal_var=False,
    alternative='greater'
)

print("Estadístico:", statistic)
print("P-valor:", p_value)

Estadístico: 37.992330214201516
P-valor: 1.5032478884295899e-301


Con una significación del 5 %, el t-test de Welch unilateral proporciona evidencia de que las zonas cercanas a alguno de los dos puntos definidos tienen una media de median_house_value superior a las zonas alejadas: 246.951,98 frente a 180.678,44, una diferencia de aproximadamente 66.274.
Este resultado muestra una asociación, no causalidad. No demuestra que la proximidad a la escuela o al hospital provoque precios mayores, ya que pueden influir otras características de las zonas. Además, el contraste supone independencia entre observaciones, algo que podría verse afectado por la proximidad geográfica.